# Step 2 — Silver: Quality Pre-Checks + Liquid Clustering (the shift-left core)

**Purpose:** Enforce data quality **on input** and lay data out optimally **on write** — in one
pass — so there is *no separate validation job and no separate `OPTIMIZE`/`ZORDER` job* downstream.

### The two shift-left moves
1. **Quality pre-checks (DLT expectations):** each row is tested against a rule set as it enters
   Silver. Passing rows flow through; failing rows are **quarantined** to a side table (not
   silently dropped) so nothing is lost and every rejection is explainable.
2. **Automatic Liquid Clustering on write:** the Silver table is declared with `CLUSTER BY AUTO`
   — Databricks picks and *evolves* the clustering keys from query patterns. No nightly
   `OPTIMIZE` + `ZORDER` job, and no keys to hand-pick or babysit.

### Source & Targets
| Direction | Table |
|-----------|-------|
| Source | `workspace.bronze.customers_bronze` |
| Target (Silver, auto-clustered) | `workspace.silver.customers_silver` |
| Target (quarantine) | `workspace.silver.customers_quarantine` |

> **Prerequisites:** Run `01_customers_bronze` first.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BRONZE_TABLE     = 'workspace.bronze.customers_bronze'
SILVER_TABLE     = 'workspace.silver.customers_silver'
QUARANTINE_TABLE = 'workspace.silver.customers_quarantine'

# Quality expectations as DATA (rule name -> SQL boolean). Same rule set you would pass to
# @dlt.expect_all_or_drop in a real DLT pipeline (see final cell).
EXPECTATIONS = {
    'valid_customer':    'customer_id IS NOT NULL',
    'valid_age':         'age BETWEEN 18 AND 100',
    'positive_spend':    'total_spend > 0',
    'positive_items':    'items_purchased > 0',
    'valid_rating':      'avg_rating BETWEEN 1 AND 5',
    'known_membership':  "membership_type IN ('Gold', 'Silver', 'Bronze')",
    'has_satisfaction':  'satisfaction_level IS NOT NULL',
}

spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.silver')
spark.sql(f'DROP TABLE IF EXISTS {SILVER_TABLE}')
spark.sql(f'DROP TABLE IF EXISTS {QUARANTINE_TABLE}')

print(f'Source: {BRONZE_TABLE}')
print(f'Targets: {SILVER_TABLE} (+ quarantine {QUARANTINE_TABLE})')

In [0]:
from pyspark.sql import functions as F

---

### Read Bronze

In [0]:
bronze_df = spark.read.table(BRONZE_TABLE)

print(f'Bronze rows: {bronze_df.count()}')

---

### Step 1 — Apply expectations on input (split valid vs. quarantine)

Evaluate the rule set per row, route passing rows onward, and capture failing rows with a
`failed_rules` array explaining *why* each was rejected. A rule that evaluates to NULL counts
as a **failure**, so `is_valid` is never null and every quarantined row lists at least one reason.

In [0]:
# A row passes a rule only when it evaluates to TRUE; NULL counts as a failure.
def passes(rule):
    return F.expr(rule).eqNullSafe(F.lit(True))

# Combined pass condition = passes ALL expectations
valid_cond = F.lit(True)
for rule in EXPECTATIONS.values():
    valid_cond = valid_cond & passes(rule)

# Per-row list of the expectations a row failed (NULL-safe)
fail_flags = [F.when(~passes(rule), F.lit(name)) for name, rule in EXPECTATIONS.items()]

tagged_df = (
    bronze_df
    .withColumn('is_valid',     valid_cond)
    .withColumn('failed_rules', F.array_except(F.array(*fail_flags), F.array(F.lit(None))))
)

# Valid rows: keep one row per customer (customer_id is the primary key)
valid_df = (
    tagged_df
    .filter(F.col('is_valid'))
    .drop('is_valid', 'failed_rules')
    .dropDuplicates(['customer_id'])
)

# Quarantine rows: everything that did not pass, with failure reasons attached
quarantine_df = (
    tagged_df
    .filter(~F.col('is_valid'))
    .withColumn('quarantined_at', F.current_timestamp())
)

print(f'Valid rows:       {valid_df.count()}')
print(f'Quarantined rows: {quarantine_df.count()}')

---

### Step 2 — Create the auto-clustered Silver table and write

`CLUSTER BY AUTO` is declared at creation, so Delta clusters data **as it is written** and
Databricks evolves the keys over time. There is intentionally **no `OPTIMIZE`/`ZORDER` call**.

In [0]:
# Declare the Silver table with automatic Liquid Clustering
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        customer_id              INT,
        gender                   STRING,
        age                      INT,
        city                     STRING,
        membership_type          STRING,
        total_spend              DOUBLE,
        items_purchased          INT,
        avg_rating               DOUBLE,
        discount_applied         BOOLEAN,
        days_since_last_purchase INT,
        satisfaction_level       STRING,
        ingestion_timestamp      TIMESTAMP,
        source_file              STRING
    )
    USING DELTA
    CLUSTER BY AUTO
''')

(valid_df.write
    .format('delta')
    .mode('append')
    .saveAsTable(SILVER_TABLE))

print(f'Silver rows written: {spark.read.table(SILVER_TABLE).count()}')

In [0]:
# Persist rejected rows for inspection / backfill — nothing is lost
(quarantine_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(QUARANTINE_TABLE))

print(f'Quarantine rows written: {spark.read.table(QUARANTINE_TABLE).count()}')

---

### Step 3 — Validation

In [0]:
%sql
-- Confirm Liquid Clustering is enabled (note clusterByAuto / clusteringColumns; no partitions)
DESCRIBE DETAIL workspace.silver.customers_silver

In [0]:
%sql
-- Filter query: Delta uses file-level min/max stats to skip files (data skipping).
SELECT membership_type, COUNT(*) AS customers, ROUND(AVG(total_spend), 2) AS avg_spend
FROM workspace.silver.customers_silver
WHERE city = 'New York'
GROUP BY membership_type
ORDER BY avg_spend DESC

In [0]:
%sql
-- Why each row was rejected, grouped by failure signature
SELECT failed_rules, COUNT(*) AS rows_quarantined
FROM workspace.silver.customers_quarantine
GROUP BY failed_rules
ORDER BY rows_quarantined DESC

In [0]:
valid_n = spark.read.table(SILVER_TABLE).count()
quar_n  = spark.read.table(QUARANTINE_TABLE).count()
total   = valid_n + quar_n
print(f'Passed:      {valid_n}')
print(f'Quarantined: {quar_n}')
print(f'Pass rate:   {valid_n / total:.2%}')

---

### Production form — the same logic as a declarative DLT (Lakeflow) pipeline

The cell below is the **DLT pipeline source** — it runs inside a Lakeflow Declarative Pipeline,
**not** on an interactive cluster (do not "Run" it here). `cluster_by_auto=True` and
`@dlt.expect_all_or_drop(...)` collapse Steps 1–2 into one declaration: quality and layout,
both enforced on input.

In [0]:
# ===========================================================================
# DLT PIPELINE SOURCE (declarative) — runs only inside a Lakeflow/DLT pipeline.
# Reference only: not meant to execute on an interactive cluster.
# ===========================================================================
import dlt
from pyspark.sql import functions as F

RULES = {
    'valid_customer':    'customer_id IS NOT NULL',
    'valid_age':         'age BETWEEN 18 AND 100',
    'positive_spend':    'total_spend > 0',
    'positive_items':    'items_purchased > 0',
    'valid_rating':      'avg_rating BETWEEN 1 AND 5',
    'known_membership':  "membership_type IN ('Gold', 'Silver', 'Bronze')",
    'has_satisfaction':  'satisfaction_level IS NOT NULL',
}

# Automatic Liquid Clustering + quality drop, declared together on the input.
@dlt.table(
    name='customers_silver',
    comment='Quality-gated, auto-clustered customers. No separate OPTIMIZE/validation job.',
    cluster_by_auto=True,
    table_properties={'quality': 'silver'},
)
@dlt.expect_all_or_drop(RULES)               # failing rows dropped at ingestion
def customers_silver():
    return dlt.read('customers_bronze').dropDuplicates(['customer_id'])

# Quarantine variant: warn instead of drop, then route failures to a side table.
@dlt.table(name='customers_quarantine', comment='Rows that failed Silver expectations.')
@dlt.expect_all(RULES)                        # warn (keep rows), metrics still recorded
def customers_quarantine():
    cond = ' OR '.join(f'NOT ({rule})' for rule in RULES.values())
    return dlt.read('customers_bronze').filter(F.expr(cond))

In [0]:
%sql
-- After the DLT pipeline runs, expectation pass/fail counts live in the event log:
SELECT
    e.dataset             AS dataset,
    e.name                AS expectation,
    SUM(e.passed_records) AS passed,
    SUM(e.failed_records) AS failed
FROM (
    SELECT explode(from_json(
        details:flow_progress:data_quality:expectations,
        'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>'
    )) AS e
    FROM event_log(TABLE(LIVE.customers_silver))
) GROUP BY 1, 2 ORDER BY 1, 2